# Head/tail detector across domains (T4) and what the fork error is made of (T5)

`p2-tests.md` T4 and T5. Inputs are frozen in `data/`; see `data/PROVENANCE.md`, including
why the Lite producer is an orphan and what was checked on freeze.

The detector is the same untrained function in both domains
(`FishHeadTailDetector.find_head_tail_img`). Lite picks the fish under the laser dot;
Mobile has no laser and picks by score × area. Errors are % of the human snout–fork
length per frame, orientation resolved.

In [1]:
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.precision", 2)

from fishsense_cscw import headtail as ht
mob = ht.load_mobile("v2_1_5_fix")
rows = {
    "Mobile (in air), Fishial, score×area": ht.summarize(mob),
    "Lite (underwater), Fishial, laser gate": ht.summarize(ht.load_lite("fishial"), cluster="dive_id"),
    "Lite (underwater), SAM3, laser gate": ht.summarize(ht.load_lite("sam3"), cluster="dive_id"),
}
pd.DataFrame(rows)

,"Mobile (in air), Fishial, score×area","Lite (underwater), Fishial, laser gate","Lite (underwater), SAM3, laser gate"
labelled,151,167,167
predicted,148,90,117
coverage_pct,98.01,53.89,70.06
fish_at_model_input_px_p50,519.82,124.73,124.73
snout_pct_p50,1.74,2.66,1.52
snout_pct_p50_ci,"(1.3275773892387757, 2.4411001789526847)","(1.8582528687908733, 3.036472786471768)","(1.3964392614742618, 1.791964749987641)"
snout_pct_p90,9.72,13.34,4.02
fork_pct_p50,4.79,8.99,8.37
fork_pct_p50_ci,"(2.5615728726673392, 6.142634843341879)","(6.350228676261011, 13.553903338667634)","(5.208691441188092, 12.578413673868507)"
fork_pct_p90,14.47,20.44,20.74


Lite intervals resample **whole dives** (six of them); Mobile's resample frames. The
Mobile set is 151 of 750 labelled frames, from however many sessions those came from.

## Reading it — and the confound that limits it

**Snout is domain-invariant; the fork is not.** Snout p50 sits at 1.5–2.7 % of length in
every condition. Fork p50 is 4.8 % in air against 8.4–9.0 % underwater.

**Coverage cannot be attributed to domain.** Mobile's 98 % against Lite's 54–70 % looks
like "in air is easier", but the Mobile fish reaches the segmenter ~4× larger:

In [2]:
lite = ht.load_lite("fishial")
q = lambda s: s.quantile([.1, .5, .9]).round().astype(int).tolist()
print("fish length at segmenter input, px, p10/p50/p90")
print("  Mobile         :", q(mob.model_input_px))
print("  Lite (Fishial) :", q(lite.model_input_px))
lite["size_bin"] = pd.cut(lite.model_input_px, [0, 75, 125, 200, 2000])
lite.groupby("size_bin", observed=True).predicted.agg(["mean", "size"]).rename(columns={"mean": "coverage", "size": "frames"})

fish length at segmenter input, px, p10/p50/p90
  Mobile         : [340, 520, 681]
  Lite (Fishial) : [73, 125, 251]


,coverage,frames
size_bin,,
"(0, 75]",0.00,19
"(75, 125]",0.48,65
"(125, 200]",0.72,64
"(200, 2000]",0.68,19


Lite coverage climbs from 0 % below 75 px to ~70 % above 125 px — the resolution lever
`fish-detection-labeling.md` §0.4 found. Mobile sits almost entirely above any Lite
frame. The fair comparison is Lite through the SAM3 1800×1350 laser crop, which rescales
by 0.56 — close to Mobile's 0.55 — and that needs the NAS and a GPU (T9's setup).

The **fork gap survives size-matching**, on thin evidence:

In [3]:
band = lite[(lite.model_input_px >= 250) & (lite.model_input_px <= 700)]
bp = band[band.predicted]
print(f"Lite frames in Mobile's size band: {len(band)} labelled, {len(bp)} predicted, {bp.dive_id.nunique()} dives")
print(f"  snout p50 {bp.snout_pct.median():.2f} %   fork p50 {bp.fork_pct.median():.2f} %   (Mobile: 1.74 / 4.79)")

Lite frames in Mobile's size band: 17 labelled, 12 predicted, 3 dives
  snout p50 1.37 %   fork p50 9.68 %   (Mobile: 1.74 / 4.79)


Twelve frames from three dives: suggestive, not a result.

## T5 — the fork error is lateral, not a fixable offset

`headtail-prediction.md` §9.2 asks whether the fork error is a labelling convention (a
tight along-axis offset a constant would remove) or a model problem. Mobile only: the
Lite run kept error magnitudes, not coordinates.

In [4]:
d = ht.fork_decomposition(mob)
cols = ["fork_along_pct", "fork_across_pct", "snout_along_pct", "snout_across_pct",
        "mask_past_fork_pct", "mask_width_past_fork_pct"]
d[cols].describe(percentiles=[.25, .5, .75, .9]).T[["count", "25%", "50%", "75%", "90%"]]

,count,25%,50%,75%,90%
fork_along_pct,148.0,0.13,0.77,1.86,3.45
fork_across_pct,148.0,0.68,4.04,8.49,11.56
snout_along_pct,148.0,-0.24,0.38,1.09,1.89
snout_across_pct,148.0,0.43,1.23,7.41,9.14
mask_past_fork_pct,148.0,1.56,3.46,5.11,8.82
mask_width_past_fork_pct,148.0,10.17,19.09,30.30,33.93


In [5]:
import numpy as np
along = d.fork_along_pct.median()
total = np.hypot(d.fork_along_pct, d.fork_across_pct)
fixed = np.hypot(d.fork_along_pct - along, d.fork_across_pct)
print(f"frames past the notch: {(d.fork_along_pct > 0).mean():.0%}")
print(f"fork p50 {total.median():.2f} %  ->  {fixed.median():.2f} % if the median along-axis offset were subtracted")

frames past the notch: 78%
fork p50 4.79 %  ->  4.61 % if the median along-axis offset were subtracted


**Verdict: not a convention offset.** The tail lands a small, systematic 0.8 % of length
*past* the notch (78 % of frames), but the typical error is **sideways** — 4 % of length
off the axis. Subtracting a constant barely moves the fork error (4.79 → 4.61 %).

The masks say why: in air the segmentation reaches 3.5 % of length past the human fork
and is ~19 % of length wide there. The spread caudal lobes are in the mask, so the fork
notch is a concavity between two convex tips and the geometry step lands beside it —
the GEO regime `validation.ipynb` §5 already saw as "different lateral tips of the
forked caudal fin".

The fix this points to is a **notch finder** (the concave point between the lobes), not
a calibration constant. It also explains part of Mobile's +2 % signed length bias.
Whether the same holds underwater, where fins are often translucent and may not be in
the mask at all, needs the Lite coordinates.